# <font color="#418FDE" size="6.5" uppercase>**C: Generative Adversarial Networks (GANs) for Image-to-Image Translation**</font>
----

> Last update: 20240710

By the end of this lecture, you will be able to:

* Develop a Generative Adversarial Network (GAN) for an image-to-image translation experiment.  
 * Some of the codes are inspired/adopted from [Image Super-Resolution GANs](https://www.udemy.com/course/image-super-resolution-gans)


## **1. Image-to-Image Translation Using Generative Adversarial Networks (GANs)**

GANs, particularly models [Image-to-Image Transiton](https://arxiv.org/abs/1611.07004), can be used for tasks such as image denoising, image coloring, pose translation, face-to-comic conversion, etc. GANs consist of two networks: a generator & discriminator. The generator tries to create normal images from abnormal inputs, while the discriminator attempts to distinguish between real normal images & those generated by the generator. This adversarial process results in highly realistic, normal images. The terms normal & abnormal are relative to the task & represent the input image to the generator & the expected output of the generator, for example, a black & white image as abnormal & a 3-channel RGB color image as normal or noisy image as abnormal & the denoised image as normal.

### **1.1. Proposed Generators Architecture**

The following Figure depicts a generator architecture used in image-to-image translation tasks. This architecture represents a convolutional neural network (CNN) with residual blocks commonly employed in image-based models. Here’s an explanation of the key components & the flow of data through the architecture:

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_06/generator_normalabnormal_resized.png?raw=true" width="50%">
  <br>
  <figcaption>Figure: Generator Architecture for image-to-image translation where $N$ is a natural number ($N\ge5$).
</div>

> **Input Image**
* The input to the network is an abnormal image of size  $2^N \times 2^N$.

> **Residual Blocks**
* The image first passes through several residual blocks. Residual blocks are a series of layers that help the network learn the identity mapping more easily. They help in training very deep networks by allowing gradients to flow through the network without vanishing or exploding.
* Residual blocks typically consist of convolutional layers, activation functions (like ReLU), & skip connections that add the input of the block to its output.

> **Final Output**
* The final output is a normal (desired) image of size $2^{N} \times 2^{N}$.
* The goal is for this image to be a high-quality, normal version of the original abnormal input image.

### **1.2. Proposed Discriminator Architecture**

The following Figure depicts the discriminator architecture used in an image-to-image translation GAN. The discriminator’s role is to distinguish between real normal images & the normal images generated by the generator. Here’s an explanation of the key components & the flow of data through the discriminator architecture:

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_06/discriminator_normalabnormal_resized.png?raw=true" width="75%">
  <br>
  <figcaption>Figure: Discriminator Architecture for image-to-image translation task where $N$ is a natural number ($N\ge5$).
</div>

> **Input Images**
* The discriminator receives two inputs: a normal image of size $2^{N} \times 2^{N}$ & an abnormal image of size  $2^N \times 2^N$. Next, it concatenates these inputs into a 6-channel $2^{N} \times 2^{N}$ feature map.
* The normal image can either be a real image from the dataset or a generated image from the generator.

> **Convolutional Layers with Residual Blocks**
* The normal & abnormal images are processed through a series of convolutional layers, often interspersed with residual blocks.
* Each residual block typically consists of convolutional layers & activation functions (like ReLU), along with skip connections that concatenate the input of the block to its output.

> **Downsampling**
* The architecture involves progressively downsampling the 6-channel concatenated feature map through convolutional layers. These layers reduce the spatial dimensions while increasing the depth (number of channels) of the feature maps.
* The first downsampling reduces the image to $2^{N-1} \times 2^{N-1}$, followed by further downsampling to $2^{N-2} \times 2^{N-2}$, & eventually to  $4 \times 4$.

> **Skip Connections**
* The discriminator uses skip connections where intermediate feature maps from the residual blocks are concatenated at various stages.
* These connections help in retaining information from different resolutions, aiding the network in distinguishing between real & generated images more effectively.

> **Realism Score**
* After downsampling to a small spatial resolution (e.g., $4 \times 4$), the final layer of the network produces a realism score.
* The realism score is a single value indicating how real or fake the input normal image is. This score can range from $-\infty$ to $+\infty$, with higher scores indicating a higher likelihood of the image being real.

## **2. Image-to-Image Translation Using Experiment**

In this section, we will develop an image-to-image translation task for "denoising" animal faces dataset using a GAN model.

In [ ]:
#@title Required Libraries, Functions, & Classes
import os
import glob
import copy
import math
import random
import time
import shutil
import cv2

import tensorflow as tf
import numpy as np

from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *

def create_blur_filter(dtype: Any) -> tf.Tensor:
    """
    Creates a Gaussian-like blur filter tensor.

    Args:
        dtype: The data type of the returned tensor (e.g., tf.float32).

    Returns:
        tf.Tensor: A 4x4 blur filter with specified dtype.
    """
    return tf.constant(
        [
            [0.015625, 0.046875, 0.046875, 0.015625],
            [0.046875, 0.140625, 0.140625, 0.046875],
            [0.046875, 0.140625, 0.140625, 0.046875],
            [0.015625, 0.046875, 0.046875, 0.015625]
        ],
        dtype=dtype
    )

def blur(x: tf.Tensor, strides: int = 1) -> tf.Tensor:
    """
    Applies a depthwise convolution to input tensor 'x' using a Gaussian-like blur filter.
    This function blurs the input image by applying a predefined kernel that approximates
    a Gaussian blur. The blurring is done via depthwise convolution, which applies
    a single filter to each input channel independently.

    Args:
        x (tf.Tensor): The input tensor to be blurred. Typically, this tensor should
                       have a shape of [batch_size, height, width, channels].
        strides (int): The stride size for the depthwise convolution operation. Strides
                       specify how much the filter moves in each step of the convolution.
                       A stride of 1 means the filter is moved one pixel at a time.

    Returns:
        tf.Tensor: The blurred output tensor, which has the same shape as the input tensor
                   but with the spatial information smoothed by the convolution.
    """

    # Determine the number of channels in the input tensor.
    # This is necessary to properly replicate the blur filter across all channels.
    channel_count = x.shape[3]

    # Create a 4x4 Gaussian-like blur filter specific to the data type of the input tensor.
    # This filter is initially 2D (height x width), & we need to expand its dimensions
    # to apply it across each channel independently.
    filter = create_blur_filter(x.dtype)[:, :, tf.newaxis, tf.newaxis]

    # Tile the filter across the channel dimension to ensure each channel receives the
    # same filter during the depthwise convolution. This expands the filter from 2D to 4D,
    # matching the number of dimensions required for depthwise convolution.
    filter = tf.tile(filter, [1, 1, channel_count, 1])

    # Apply the depthwise convolution using the replicated filter. The convolution applies
    # the filter to each input channel independently without mixing between channels.
    # The strides parameter controls how the filter is moved across the input tensor,
    # & 'SAME' padding ensures the output tensor has the same spatial dimensions as the input.
    return tf.nn.depthwise_conv2d(x, filter, strides=[1, strides, strides, 1], padding='SAME')

def downsample(x: tf.Tensor) -> tf.Tensor:
    """
    Downsamples the input tensor 'x' by a factor of 2 using a Gaussian-like blur filter and
    a stride of 2 in the convolution.

    Args:
        x (tf.Tensor): The input tensor to be downsampled.

    Returns:
        tf.Tensor: The downsampled output tensor.
    """
    return blur(x, strides=2)

def upsample(x: tf.Tensor) -> tf.Tensor:
    """
    Upsamples the input tensor 'x' by a factor of 2 using nearest neighbor interpolation
    followed by a blur to smooth the result. This method increases the spatial resolution
    of the image by inserting zeros between pixels & then applies a blur to integrate
    these new pixels into the original image context.

    Args:
        x (tf.Tensor): The input tensor to be upsampled. Expected to have the format
                       [batch_size, height, width, channels].

    Returns:
        tf.Tensor: The upsampled & blurred output tensor, which will have double the
                   height & width of the input tensor.
    """
    def upsample_with_zeros(x: tf.Tensor) -> tf.Tensor:
        """
        Internal function to perform nearest neighbor upsampling by inserting zeros
        between the pixels of the input tensor.

        Args:
            x (tf.Tensor): Input tensor with dimensions [batch_size, height, width, channels].

        Returns:
            tf.Tensor: Tensor with dimensions [batch_size, height*2, width*2, channels], where
                       zeros have been inserted between original pixels.
        """
        # Extract dimensions from the input tensor.
        batch_size, in_height, in_width, channel_count = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]

        # Calculate the output dimensions, which are double the input dimensions.
        out_height, out_width = in_height * 2, in_width * 2

        # Reshape the input tensor to prepare for zero insertion.
        # We create extra dimensions at height & width positions & then pad these
        # dimensions with zeros to interleave the original pixels with zeros.
        x = tf.reshape(x, [batch_size, in_height, 1, in_width, 1, channel_count])
        x = tf.pad(x, [[0, 0], [0, 0], [0, 1], [0, 0], [0, 1], [0, 0]])

        # Reshape back to the expected output dimensions with the new height & width.
        return tf.reshape(x, [batch_size, out_height, out_width, channel_count])

    # Apply the upsample_with_zeros function to the input tensor, multiplying by 4 to
    # compensate for the increase in area, so that the intensity of the image remains
    # consistent. Then, apply a blur to smooth the image, reducing artifacts introduced
    # by the zero-insertion upsampling method.
    return blur(upsample_with_zeros(x * 4.))

def pixel_norm(x: tf.Tensor, epsilon: float = 1e-7) -> tf.Tensor:
    """
    Normalizes the input tensor by its pixel values to stabilize training in neural networks, particularly in GANs.

    Pixel normalization is performed per pixel over all channels. It divides each pixel by the square root of
    its mean squared value to normalize feature vectors to unit length, adding a small epsilon to the denominator
    for numerical stability to prevent division by zero.

    Args:
        x (tf.Tensor): The input tensor to normalize. Expected to have any shape with channels as the last dimension.
        epsilon (float): A small constant to ensure numerical stability by avoiding division by zero.

    Returns:
        tf.Tensor: The pixel-normalized tensor, cast back to the original data type of the input tensor.

    Notes:
        This function first casts the input tensor to float32 to ensure high precision during computation.
        The normalization uses the square root of the mean squared value across the channels, incorporating
        epsilon directly under the square root to maintain non-negative values. The result is then cast back
        to the original data type of the input tensor to maintain consistency in subsequent operations.
    """
    # Store the original data type of the input tensor to restore it after computation
    original_dtype = x.dtype

    # Cast the input tensor to float32 for high precision during division & square root operations
    x = tf.cast(x, tf.float32)

    # Compute the square of x, then average across the channels, add epsilon for numerical stability (in case of small vals)
    mean_sq = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True) + epsilon

    # Normalize the input x by the square root of the mean squared value
    normalized = x / tf.math.sqrt(mean_sq)

    # Cast the normalized tensor back to the original data type of the input tensor
    return tf.cast(normalized, original_dtype)

def reduce_std_nan_safe(x: tf.Tensor,
                        axis: Optional[int] = None,
                        keepdims: bool = False,
                        epsilon: float = 1e-7) -> tf.Tensor:
    """
    Compute the standard deviation of a tensor along the specified axes, adding a small epsilon
    for numerical stability. This function is used to avoid division by zero when the variance
    is zero.

    Args:
        x (tf.Tensor): Input tensor whose standard deviation is to be calculated.
        axis (Optional[int]): The axis or axes along which to compute the mean & variance.
                              If None, standard deviation is computed over the whole tensor.
        keepdims (bool): If True, retains reduced dimensions with length 1 in the result.
        epsilon (float): A small constant added to the variance to improve numerical stability.

    Returns:
        tf.Tensor: A tensor containing the computed standard deviation, cast to the original
                   data type of input tensor `x`.

    Notes:
        The function first casts the input to float32 for stability in computations involving
        the mean & variance. It then calculates the mean of the tensor along the specified
        axis. The variance is determined as the average of the squared deviations from the mean.
        Adding a small epsilon before taking the square root ensures non-negative results, guarding
        against numerical issues. Finally, the result is cast back to the dtype of the input tensor
        to maintain consistency in data type.
    """
    # Cast input tensor to float32 to ensure stability in calculations
    y = tf.cast(x, tf.float32)

    # Compute the mean of the elements in the tensor along the specified axis
    mean = tf.reduce_mean(y, axis=axis, keepdims=True)

    # Calculate the variance as the mean of squared deviations from the mean
    variance = tf.reduce_mean(tf.square(y - mean), axis=axis, keepdims=keepdims)

    # Compute the standard deviation as the square root of variance, adding epsilon for numerical stability
    sqrt = tf.sqrt(variance + epsilon)

    # Cast the result back to the original data type of the input tensor
    return tf.cast(sqrt, x.dtype)

def minibatch_standard_deviation(x: tf.Tensor,
                                 group_size: int = 4) -> tf.Tensor:
    """
    Adds a minibatch standard deviation feature map to the last dimension of the input tensor.

    This function helps a neural network to take into account the variance of the minibatch, allowing
    it to learn from the internal statistics of the samples in each batch. It is often used in
    generative models like GANs to improve model stability & convergence.

    Args:
        x (tf.Tensor): Input tensor of shape (N, H, W, C) where:
            N = batch size
            H = height
            W = width
            C = channels
        group_size (int): The size of the groups into which the batch is split. Default is 4.

    Returns:
        tf.Tensor: Tensor with an additional feature map appended to the last channel, resulting in
                   the shape (N, H, W, C + 1).
    """
    # Get the original shape & data type of the input tensor
    original_shape = tf.shape(x)
    original_dtype = x.dtype

    # Compute the number of possible groups
    global_sample_count = original_shape[0]
    # Make sure we have at least three images; otherwise, the standard deviation is not working well.
    group_size = tf.minimum(group_size, global_sample_count)
    group_count = global_sample_count // group_size

    # Ensure the total number of elements is exactly divisible by the group size
    tf.Assert(
        group_size * group_count == global_sample_count,
        ['Sample count was not divisible by group size']
    )

    # Reshape the input to group the batch dimension into (group_count, group_size)
    y = tf.reshape(
        x,
        tf.concat([[group_count, group_size], original_shape[1:]], axis=0)
    )
    y = tf.cast(y, tf.float32)

    # Compute the standard deviation within each group
    stddevs = reduce_std_nan_safe(y, axis=1, keepdims=True)

    # Calculate the mean standard deviation across all groups & features
    avg_stddev = tf.reduce_mean(
        stddevs,
        axis=tf.range(1, tf.rank(stddevs)),
        keepdims=True
    )

    # Create a new feature map with the same spatial dimensions & append to the last channel
    new_feature_shape = tf.concat([tf.shape(y)[:-1], [1]], axis=0)
    new_feature = tf.broadcast_to(avg_stddev, new_feature_shape)
    y = tf.concat([y, new_feature], axis=-1)

    # Reshape back to the original batch dimension while preserving the new feature channel
    y = tf.reshape(
        y,
        tf.concat([[global_sample_count], original_shape[1:-1], [original_shape[-1] + 1]], axis=0)
    )
    y = tf.cast(y, original_dtype)

    return y

def count_records_in_tfrecord(tfrecord_path: str) -> int:
    """
    Count the number of records in a single TFRecord file.

    This function iterates through each record in the given TFRecord file & increments
    a counter to determine the total number of records.

    Args:
        tfrecord_path (str): The file path or URI to the TFRecord file. This can be a local
                             path or a remote path in a storage service like Google Cloud Storage
                             (indicated by a URI like gs://bucket_name/path_to_file).

    Returns:
        int: The total number of records in the specified TFRecord file.
    """
    count = 0
    # Initialize a TFRecordDataset to read from the specified file
    for _ in tf.data.TFRecordDataset(tfrecord_path):
        count += 1
    return count

def count_images_in_directory(directory_path: str) -> int:
    """
    Count the total number of images across all TFRecord files in a specified directory.

    This function first retrieves a list of all TFRecord files in the given directory,
    including support for remote directories in cloud storage. It then iterates through
    each TFRecord file, counting the records using `count_records_in_tfrecord`, and
    sums these counts to get the total number of images.

    Args:
        directory_path (str): The directory path containing TFRecord files. This can be a local
                              directory path or a remote directory path in a storage service
                              like Google Cloud Storage (e.g., "gs://bucket_name/path_to_directory/").

    Returns:
        int: The total number of records (images) across all TFRecord files in the directory.
    """
    total_count = 0
    # Use tf.io.gfile.glob to support both local & remote filesystems (e.g., GCS)
    tfrecord_files = tf.io.gfile.glob(os.path.join(directory_path, '*.tfrecord') )
    # Iterate through each file & count the records within
    for tfrecord_file in tfrecord_files:
        file_count = count_records_in_tfrecord(tfrecord_file)
        total_count += file_count
    return total_count

def generator_residual_block(x_0: tf.Tensor, reduced_factor: float) -> tf.Tensor:
    """
    Applies a series of residual blocks to the input tensor to generate a transformed output tensor.

    Args:
        x_0 (tf.Tensor): Input tensor.
        reduced_factor (float): Factor to reduce the number of channels in the convolution layers.

    Returns:
        tf.Tensor: Transformed output tensor after applying the residual blocks.
    """
    for _ in range(5):
        # Copy the input tensor to a new variable
        x = x_0

        # Determine the number of channels for the convolution layers
        channel_count = int(max(32, min(512 / reduced_factor, x.shape[-1] * 2)))

        # Apply two convolutional layers with LeakyReLU activation
        for _ in range(2):
            x = ScaledConv2d(channel_count=channel_count, kernel_size=3, padding='same')(x)
            x = ScaledLeakyReLU()(x)

        # Copy the original input tensor
        x_1 = x_0

        # If the channel count does not match, apply a 1x1 convolution to match dimensions
        if x_1.shape[-1] != channel_count:
            x_1 = ScaledConv2d(channel_count=channel_count, kernel_size=1, padding='same')(x_1)
            x_1 = ScaledLeakyReLU()(x_1)

        # Add the original & the transformed tensors
        x = ScaledAdd()([x, x_1])

        # Update the input tensor for the next residual block
        x_0 = x

    # Return the final transformed tensor
    return x_0

def discriminator_conv_block(filter_size: Tuple[int, int], x: tf.Tensor) -> tf.Tensor:
    """
    Applies a series of convolutional & activation layers to the input tensor for the discriminator.

    Args:
        filter_size (Tuple[int, int]): A tuple containing the number of filters for the convolution layers.
        x (tf.Tensor): Input tensor.

    Returns:
        tf.Tensor: Transformed output tensor after applying the convolutional block.
    """
    # Apply the first convolutional layer with the first filter size
    x0 = ScaledConv2d(channel_count=filter_size[0], kernel_size=3, strides=1, padding='same')(x)

    # Apply the first LeakyReLU activation layer
    x1 = ScaledLeakyReLU()(x0)

    # Apply the second convolutional layer with the second filter size
    x2 = ScaledConv2d(channel_count=filter_size[1], kernel_size=3, strides=2, padding='same')(x1)

    # Apply the second LeakyReLU activation layer
    x3 = ScaledLeakyReLU()(x2)

    # Downsample the input tensor
    x4 = Downsample()(x)

    # Apply the third convolutional layer with the second filter size
    x5 = ScaledConv2d(channel_count=filter_size[1], kernel_size=3, strides=1, padding='same')(x4)

    # Add the outputs of the third convolutional layer & the downsampled tensor
    return ScaledAdd()([x3, x5])

class Upsample(tf.keras.layers.Layer):
    """
    A custom Keras layer that upsamples the input tensor using a predefined upsampling function.

    This layer acts as a wrapper that applies an upsampling operation defined outside of this class.
    It's intended to increase the spatial dimensions (height & width) of the input tensor.

    Methods:
        call(x): Performs the upsampling operation on the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Call method for the upsampling layer.

        Args:
            x (tf.Tensor): Input tensor to the layer.

        Returns:
            tf.Tensor: The upsampled tensor.

        Notes:
            This method assumes that an `upsample` function is defined elsewhere in the codebase,
            which should handle the specifics of the upsampling operation (e.g., using nearest
            neighbor interpolation followed by a blur for smoothing).
        """
        return upsample(x)  # Assumes 'upsample' is defined elsewhere with the desired logic.

class Downsample(tf.keras.layers.Layer):
    """
    A custom Keras layer that downsamples the input tensor using a predefined downsampling function.

    This layer acts as a wrapper that applies a downsampling operation defined outside of this class.
    It's intended to reduce the spatial dimensions (height & width) of the input tensor.

    Methods:
        call(x): Performs the downsampling operation on the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Call method for the downsampling layer.

        Args:
            x (tf.Tensor): Input tensor to the layer.

        Returns:
            tf.Tensor: The downsampled tensor.

        Notes:
            This method assumes that a `downsample` function is defined elsewhere in the codebase,
            which should handle the specifics of the downsampling operation (e.g., using a blur
            followed by a stride-based convolution to reduce dimensions).
        """
        return downsample(x)  # Assumes 'downsample' is defined elsewhere with the desired logic.

class ScaledLeakyReLU(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies the Leaky ReLU activation function with a scaling factor.

    Leaky ReLU is an activation function defined as:
        f(x) = alpha * x for x < 0,
        f(x) = x for x >= 0,
    where `alpha` is a small coefficient. This layer further scales the output by a constant
    `gain` to control the magnitude of the output, which can be useful in maintaining
    the neural network's training dynamics.

    Attributes:
        alpha (float): Coefficient of leakage in the negative part of the function.
        gain (float): Scaling factor applied to the output of the activation function.
    """

    def __init__(self, alpha: float = 0.2, gain: float = math.sqrt(2.0), **kwargs):
        """
        Initialize the ScaledLeakyReLU layer.

        Args:
            alpha (float): Coefficient for the negative slope of the Leaky ReLU.
            gain (float): Gain factor to scale the output of the activation.
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gain = gain

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Logic for the layer's forward pass.

        Args:
            inputs (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Output tensor after applying the Leaky ReLU activation & scaling.
        """
        return tf.nn.leaky_relu(inputs, self.alpha) * self.gain

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary for serialization.

        Returns:
            Dict[str, Any]: Configuration of the layer including `alpha` & `gain`.
        """
        config = super().get_config()
        config.update({
            'alpha': self.alpha,
            'gain': self.gain
        })
        return config

class ScaledConv2d(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a scaled 2D convolution to the input (performs downsampling).

    This layer introduces a scaling factor to the convolution operation to manage the amplitude
    of outputs, potentially improving the stability & performance of neural networks.
    Optionally, a blur operation can be applied to the inputs before the convolution to reduce
    high-frequency noise.

    Attributes:
        channel_count (int): The number of output channels (filters) in the convolution.
        kernel_size (int): The dimensions of the convolution window (height & width).
        strides (int): The number of pixels by which the convolution window moves during the sliding.
        padding (str): The strategy for handling the border of the input ('valid' for no padding,
                       'same' to pad input so the output has the same width/height dimension).
        pre_blur (bool): A flag to determine whether to blur the input before the convolution.
    """

    def __init__(
        self,
        channel_count: int,
        kernel_size: int,
        strides: int = 1,
        padding: str = 'valid',
        pre_blur: bool = False,
        **kwargs
    ):
        """
        Initialize the ScaledConv2d layer with necessary configurations.

        Args:
            channel_count (int): Number of filters in the convolution.
            kernel_size (int): Size of the convolutional kernel (e.g., 3 for a 3x3 kernel).
            strides (int): Stride of the convolution.
            padding (str): Padding type, either 'valid' (no padding) or 'same' (pad to keep size).
            pre_blur (bool): Whether to pre-blur the inputs to soften features.
            **kwargs: Additional keyword arguments for the base Layer class.
        """
        super().__init__(**kwargs)
        # Normalizing parameters to ensure they are in acceptable formats.
        self.rank = 2  # Convolutional rank indicating it's a 2D convolution.
        self.channel_count = channel_count
        self.kernel_size = conv_utils.normalize_tuple(kernel_size, self.rank, 'kernel_size')
        self.strides = conv_utils.normalize_tuple(strides, self.rank, 'strides')
        self.padding = conv_utils.normalize_padding(padding)
        self.pre_blur = pre_blur

    def build(self, input_shape: List[int]) -> None:
        """
        Build the weights of the layer.

        Args:
            input_shape (List[int]): Shape of the input tensor, expected to include channel dimension.

        Raises:
            AssertionError: If the provided input shape does not meet the expected criteria.
        """
        # Checking to ensure the input shape is correct.
        assert len(input_shape) == self.rank + 2, "Input shape must match rank 2 plus batch & channel dimensions."

        # Configuring the shape of the weights based on input dimensions & the number of output channels.
        in_channel_count = input_shape[-1]
        kernel_shape = self.kernel_size + (in_channel_count, self.channel_count)
        self.kernel = self.add_weight(
            name='kernel',
            shape=kernel_shape,
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.channel_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        # Scaling factor to adjust the amplitude of the output, thereby helping control the learning dynamics.
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / tf.sqrt(tf.reduce_prod(tf.cast(kernel_shape[:-1], tf.float32)))),
            trainable=False
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Perform the convolution operation on input data.

        Args:
            inputs (tf.Tensor): Input tensor to be processed by convolution.

        Returns:
            tf.Tensor: The output tensor after applying the convolution & bias.
        """
        y = inputs
        if self.pre_blur:
            # Optionally apply a blur to the input to reduce noise & soften features.
            y = self.blur(y)  # Note: 'blur' should be a defined function or method in this or imported context.
        # Apply convolution with the scaled kernel & appropriate padding & stride settings.
        y = tf.nn.conv2d(
            y,
            self.kernel * self.scale,
            strides=self.strides,
            padding=self.padding.upper()
        )
        # Add the bias to the convoluted outputs.
        y = tf.nn.bias_add(y, self.bias)
        return y

    def get_config(self) -> Dict[str, Any]:
        """
        Serialize the configuration of the layer for storage or copying.

        Returns:
            Dict[str, Any]: A dictionary containing all configuration details of the layer.
        """
        config = super().get_config()
        config.update({
            'channel_count': self.channel_count,
            'kernel_size': self.kernel_size,
            'strides': self.strides,
            'padding': self.padding,
            'pre_blur': self.pre_blur
        })
        return config

class UpsampleConv2d(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a transposed convolution to upsample the input tensor.

    This layer increases the spatial dimensions of the input tensor using a transposed convolution,
    often referred to as a fractionally-strided convolution or a deconvolution. This operation is
    useful for tasks like image super-resolution, image-to-image translation, segmentation, & generative modeling where
    increasing image resolution or volume size is required.

    Attributes:
        channel_count (int): The number of output channels in the convolution.
    """

    def __init__(
        self,
        channel_count: int,
        **kwargs
    ):
        """
        Initializes the UpsampleConv2d layer with specified number of output channels.

        Args:
            channel_count (int): Number of output filters in the transposed convolution.
            **kwargs: Arbitrary keyword arguments for the base Layer class.
        """
        super().__init__(**kwargs)
        self.rank = 2  # Denotes 2D convolutional layers.
        self.channel_count = channel_count
        # Normalizing the kernel size to 3x3 & strides to 2 for upsampling.
        self.kernel_size = conv_utils.normalize_tuple(3, self.rank, 'kernel_size')
        self.strides = conv_utils.normalize_tuple(2, self.rank, 'strides')
        # Using 'same' padding to keep the output size as expected after convolution.
        self.padding = conv_utils.normalize_padding('same')

    def build(self, input_shape: List[int]) -> None:
        """
        Build the weights of the layer based on the input shape.

        Args:
            input_shape (List[int]): Shape of the input tensor.

        Raises:
            AssertionError: If the input shape does not have the expected dimensions.
        """
        assert len(input_shape) == self.rank + 2, "Input shape must be 4D (batch, height, width, channels)."
        in_channel_count = input_shape[-1]
        # Defining the shape of the kernel for the transposed convolution.
        kernel_shape = self.kernel_size + (self.channel_count, in_channel_count)
        self.kernel = self.add_weight(
            name='kernel',
            shape=kernel_shape,
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.channel_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        # Scale to adjust the amplitude of the outputs.
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / tf.sqrt(tf.cast(tf.reduce_prod(kernel_shape) // self.channel_count, tf.float32))),
            trainable=False
        )

    def compute_output_shape(self, input_shape):
        """
        Compute the output shape of the layer given the input shape.

        Args:
            input_shape: Shape of the input tensor.

        Returns:
            The expected output shape after the transposed convolution.
        """
        input_shape = tf.TensorShape(input_shape).as_list()
        # Output height & width are doubled due to the stride of 2.
        return tf.TensorShape([
            input_shape[0],
            None if input_shape[1] is None else input_shape[1] * 2,
            None if input_shape[2] is None else input_shape[2] * 2,
            self.channel_count])

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Perform the transposed convolution on the input tensor.

        Args:
            x (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Upsampled tensor after applying transposed convolution & adding bias.
        """
        input_shape = tf.shape(x)
        batch_size, in_height, in_width = input_shape[0], input_shape[1], input_shape[2]
        output_shape = (batch_size, in_height * 2, in_width * 2, self.channel_count)

        # Perform the transposed convolution.
        y = tf.nn.conv2d_transpose(
            x,
            self.kernel * self.scale,
            output_shape,
            self.strides,
            padding=self.padding.upper()
        )

        # Ensure the shape is set during graph execution.
        if not tf.executing_eagerly():
            y.set_shape(self.compute_output_shape(x.shape))

        # Add the bias to the output.
        y = tf.nn.bias_add(y, self.bias)
        # Apply a blur for smoothing the output, assuming 'blur' is a defined function.
        y = blur(y) # removes the checkerboard artifact; see more at https://distill.pub/2016/deconv-checkerboard/
        return y

    def get_config(self) -> Dict[str, Any]:
        """
        Serialize the configuration of the layer to allow for model saving, loading, & cloning.

        Returns:
            A dictionary containing all configuration details of the layer.
        """
        config = super().get_config()
        config.update({
            'channel_count': self.channel_count
        })
        return config

class ScaledDense(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a scaled dense (fully connected) operation to the input.

    This layer creates a dense layer where the weights are scaled by a calculated factor based on
    the number of input units. This helps in maintaining a balanced variance across different sizes
    of input dimensions.

    Attributes:
        output_count (int): Number of neurons in the dense layer.
    """

    def __init__(
        self,
        output_count: int,
        **kwargs
    ):
        """
        Initialize the ScaledDense layer.

        Args:
            output_count (int): Number of output units (neurons) in the layer.
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.output_count = output_count

    def build(self, input_shape: List[int]) -> None:
        """
        Create the layer's weights.

        Args:
            input_shape (List[int]): Shape of the input tensor to the layer, should be 2D.

        Raises:
            AssertionError: If the input shape is not 2D.
        """
        assert len(input_shape) == 2, "Input shape must be 2D (batch_size, features)"
        self.kernel = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.output_count),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.output_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / math.sqrt(input_shape[-1])),
            trainable=False
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Logic for the layer's forward pass.

        Args:
            inputs (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Output tensor after applying the scaled dense transformation & bias.
        """
        y = tf.matmul(inputs, self.kernel * self.scale)
        return tf.nn.bias_add(y, self.bias)

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary for serialization.

        Returns:
            Dict[str, Any]: Configuration of the layer.
        """
        config = super().get_config()
        config.update({
            'output_count': self.output_count
        })
        return config

class ScaledAdd(tf.keras.layers.Layer):
    """
    A custom Keras layer that performs a scaled addition of two input tensors.

    This layer first adds two equally shaped tensors element-wise, & then scales
    the result by a constant factor. The scaling factor is a non-trainable weight
    initialized to a given value.

    Attributes:
        scale_value (float): The initial scaling factor applied to the summed inputs.
    """

    def __init__(self, scale: float = 1.0 / math.sqrt(2.0), **kwargs):
        """
        Initialize the ScaledAdd layer.

        Args:
            scale (float): The scaling factor for the addition operation. Default is
                           1/sqrt(2) to maintain a similar scale in transformations.
            **kwargs: Arbitrary keyword arguments for base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.scale_value = scale

    def build(self, input_shapes: Tuple[tf.TensorShape, tf.TensorShape]) -> None:
        """
        Create the layer's weights.

        Args:
            input_shapes (tuple of tf.TensorShape): Shapes of the two inputs. Both inputs
                                                   must have the same shape.

        Raises:
            AssertionError: If the input shapes do not match.
        """
        assert len(input_shapes) == 2, "ScaledAdd layer expects two inputs"
        a_shape, b_shape = input_shapes
        assert a_shape[1:] == b_shape[1:], f"Input shapes must match: {a_shape} != {b_shape}"

        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(value=self.scale_value),
            trainable=False
        )

    def call(self, inputs: List[tf.Tensor]) -> tf.Tensor:
        """
        The logic of the layer which performs the operations on the inputs.

        Args:
            inputs (list of tf.Tensor): A list containing two tensors to be added.

        Returns:
            tf.Tensor: The result of the scaled addition of the two input tensors.
        """
        return (inputs[0] + inputs[1]) * self.scale

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary.

        Returns:
            dict: A dictionary containing the configuration of the layer.
        """
        config = super().get_config()
        config.update({'scale': self.scale_value})
        return config

class MinibatchStandardDeviation(tf.keras.layers.Layer):
    """
    A custom Keras layer that adds a minibatch standard deviation feature map to the last dimension
    of the input tensor.

    This layer leverages the `minibatch_standard_deviation` function to add statistical variance
    features from the minibatch to the input tensor. This technique is commonly used in Generative
    Adversarial Networks (GANs) to enhance training stability & model convergence.

    Example:
        # Assuming `minibatch_standard_deviation` is defined elsewhere & imported.
        model = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
            MinibatchStandardDeviation()
        ])
    """

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Call method for the layer which gets invoked during model execution.

        Args:
            inputs (tf.Tensor): A 4D input tensor with shape (batch_size, height, width, channels).

        Returns:
            tf.Tensor: The input tensor with an added minibatch standard deviation feature map.
                       The resulting tensor shape is (batch_size, height, width, channels + 1).
        """
        # Call the externally defined function to add minibatch stddev feature to the input
        return minibatch_standard_deviation(inputs)

class PixelNorm(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies pixel normalization to an input tensor.

    Pixel normalization is a technique often used in generative adversarial networks (GANs) to
    stabilize the training process. It normalizes the feature vectors of the input tensor to
    unit length per pixel across the channels. This normalization helps manage scale discrepancies
    that might arise during training.

    Methods:
        call(x): Applies pixel normalization to the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Apply the pixel normalization operation to the input tensor when the layer is called
        in a model.

        Args:
            x (tf.Tensor): Input tensor to be normalized.

        Returns:
            tf.Tensor: Pixel-normalized tensor. The normalization is performed per pixel across
                       the channels, scaling each feature vector to have a unit length.

        Notes:
            This method utilizes the 'pixel_norm' function. The function computes the normalization using the square root of
            the mean squared values of the tensor's channels, effectively scaling the pixel values
            to maintain unit length across channels.
        """
        return pixel_norm(x)  # Assumes 'pixel_norm' is defined with proper functionality.

class ImageConversion(tf.keras.layers.Layer):
    """
    A custom Keras layer for converting image data between different normalization ranges.

    This layer facilitates the conversion of image pixel values between two normalization
    schemes: [0, 1] & [-1, 1]. Depending on the `conversion_mode` specified during initialization,
    the layer can either normalize images (from [0, 1] to [-1, 1]) or denormalize them (from [-1, 1] to [0, 1]).

    Attributes:
        conversion_mode (int): Indicator of the conversion type to be applied. It supports:
            - -1 for [0, 1] to [-1, 1] conversion.
            - 0 for [-1, 1] to [0, 1] conversion.

    Methods:
        call(image): Applies the conversion to the image based on the `conversion_mode`.
        get_config(): Returns the configuration of the layer, including its `conversion_mode`.
    """

    def __init__(self, conversion_mode: int, **kwargs):
        """
        Initialize the ImageConversion layer with the specified mode of conversion.

        Args:
            conversion_mode (int): Mode of the conversion. -1 for normalizing to [-1, 1] and
                                    0 for denormalizing to [0, 1].
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.conversion_mode = conversion_mode

    def call(self, image: tf.Tensor) -> tf.Tensor:
        """
        Apply the specified image conversion when the layer is called.

        Args:
            image (tf.Tensor): Input tensor representing the image, with values expected to be in
                               the appropriate range based on the conversion mode.

        Returns:
            tf.Tensor: The converted image tensor.

        Raises:
            ValueError: If an unknown conversion mode is provided.
        """
        if self.conversion_mode == -1:
            return image * 2.0 - 1.0  # Convert from [0, 1] to [-1, 1]
        elif self.conversion_mode == 0:
            return image * 0.5 + 0.5  # Convert from [-1, 1] to [0, 1]
        else:
            # Instead of asserting, raise a ValueError for undefined conversion modes
            raise ValueError(f'Unknown conversion mode: {self.conversion_mode}')

    def get_config(self):
        """
        Overrides the default method to include the `conversion_mode` in the configuration.

        Returns:
            dict: Configuration dictionary containing the `conversion_mode` & base configuration.
        """
        config = super().get_config()
        config.update({
            'conversion_mode': self.conversion_mode
        })
        return config



In [ ]:
#@title Class - Args for training
'''
No required library.
'''

class Args():
    def __init__(self):
        """
        Initializes the configuration settings for a GAN image-to-image translation training session.

        This class sets up various configuration parameters that control the training process, model architecture,
        & operational settings such as data handling & output management. These settings are crucial for customizing
        the behavior of the GAN according to specific experimental needs.
        """

        '''
        General Mandatory
        '''

        # Name of the experiment, used for organizing outputs & logs.
        self.experiment_name = 'animal_faces'
        # Type of the project
        self.project_type = 'denoising'
        # The project patent folder
        self.project_parent_folder = '/content/drive/MyDrive'

        '''
        General Optional
        '''

        # The number of samples per batch to be used during training.
        self.batch_size = 16
        # Total number of epochs to train the models.
        self.epochs = 500
        # Buffer size for shuffling the dataset. Larger sizes can improve shuffling quality but use more memory.
        self.buffer_size = 1000
        # Interval in batches to plot the generator output during training, helps monitor progress visually.
        self.plot_intervals = 50
        # Interval in batches to save checkpoints of the model weights.
        self.checkpoint_intervals = 50
        # Source image size (N).
        self.image_size = 128 # supported sizes are 2^n where n = [5,6,...,10]
        # Noise standard deviation (recommended value: less than 127.5 to simulate noise)
        self.noise_std = 75.0
        # If true, removes existing data from the folders like checkpoints & samples before starting new training.
        self.folder_remove = False
        # If true, forces the inference to run on CPU instead of GPU.
        self.gpu = True


        '''
        Generator
        '''

        # Factor by which to reduce the filter size of each Conv layer from a baseline setting, affecting model capacity (2^n).
        self.generator_reduce_factor = 1 # acceptable values: 1, 2, 4, 8
        # Learning rate for the generator's optimizer.
        self.generator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the generator.
        self.generator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the generator.
        self.generator_beta2 = 0.99

        '''
        Discriminator
        '''

        # Factor by which to reduce the filter size of each Conv layer in the discriminator, similar to generator's setting (2^n).
        self.discriminator_reduce_factor = 2 # acceptable values: 1, 2, 4, 8
        # Learning rate for the discriminator's optimizer.
        self.discriminator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the discriminator.
        self.discriminator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the discriminator.
        self.discriminator_beta2 = 0.99
        # Interval at which to apply regularization techniques during discriminator training.
        self.discriminator_regularization_interval = 16


In [ ]:
#@title Functions - Initiation

def initiate(args: Args) -> Dict[str, Any]:
    """
    Initializes the configuration for a GAN image-to-image translation training session based on provided arguments.

    Args:
        args (Args): An instance of the Args class containing all necessary configuration parameters.

    Returns:
        Dict[str, Any]: A dictionary containing structured configuration settings for both the generator & discriminator,
                        as well as other operational settings necessary for the training or inference of the GAN image-to-image translation.
    """

    # Initialize an empty dictionary to store configurations.
    opt                  = {}

    # Pass the image size
    opt['image_size'] = args.image_size

    # Pass the noise standard deviation
    opt['noise_std'] = args.noise_std

    # Initialize sub-dictionaries for generator & discriminator configurations.
    opt['generator']     = {}
    opt['discriminator'] = {}

    # Define feature counts (discriminator) & channel counts (generator) for different image sizes.
    feature_counts = {8192: (4, 8), 4096: (8, 16), 2048: (16,32), 1024: (32, 64), 512: (64, 128), 256: (128, 256),
                      128: (256, 512), 64: (512, 512), 32: (512, 512), 16: (512, 512), 8: (512, 512), 4: (512, 512)}
    channel_counts = {4: 512, 8: 512, 16: 512, 32: 512, 64: 512, 128: 256,
                      256: 128, 512: 64, 1024: 32, 2048: 16, 4096:8, 8192:4}

    # Adjust feature counts for the discriminator based on the reduction factor.
    opt['discriminator']['feature_counts'] = {key: (int(max(item[0]/args.discriminator_reduce_factor, 4)), int(max(item[1]/args.discriminator_reduce_factor, 4)))
                                              for key, item in feature_counts.items()}

    # Adjust channel counts for the generator based on reduction factor.
    opt['generator']['channel_counts'] = {key: int(max(item/args.generator_reduce_factor, 4))
                                          for key, item in channel_counts.items()}

    # Get reduced factors
    opt['generator']['reduce_factor'] = args.generator_reduce_factor
    opt['discriminator']['reduce_factor'] = args.discriminator_reduce_factor

    # Set learning rate, beta1, & beta2 parameters for the generator based on provided arguments.
    opt['generator']['lr']              = args.generator_lr
    opt['generator']['beta1']           = args.generator_beta1
    opt['generator']['beta2']           = args.generator_beta2

    # Set the regularization interval for the discriminator.
    opt['discriminator']['regularization_interval'] = args.discriminator_regularization_interval
    # Calculate & set the lazy ratio for the discriminator regularization.
    opt['discriminator']['lazy_ratio']              = opt['discriminator']['regularization_interval'] / (opt['discriminator']['regularization_interval'] + 1)
    # Set learning rate, beta1, & beta2 parameters for the discriminator.
    opt['discriminator']['lr']                      = args.discriminator_lr
    opt['discriminator']['beta1']                   = args.discriminator_beta1
    opt['discriminator']['beta2']                   = args.discriminator_beta2
    # Set batch size, epochs, number of batches, buffer size, & intervals for plotting & checkpointing.
    opt['batch_size']           = args.batch_size
    opt['epochs']               = args.epochs
    opt['buffer_size']          = args.buffer_size
    opt['plot_intervals']       = args.plot_intervals
    opt['checkpoint_intervals'] = args.checkpoint_intervals
    # Set paths to the experiment-related folders.
    # experiment path which depends on the image_size.
    opt['dir_experiment'] = os.path.join(args.project_parent_folder, args.experiment_name, str(args.image_size), args.project_type)
    # Directory where the training data is located. This should be the path to a folder containing the training records.
    opt['dir_tfrecords'] = os.path.join(args.project_parent_folder, args.experiment_name, str(args.image_size), 'tfrecords')
    # Checkpoints path of discriminator & generators.
    opt['dir_checkpoints'] = os.path.join(opt['dir_experiment'], 'checkpoints')
    # Path to where sample-generated images are saved during training.
    opt['dir_samples'] = os.path.join(opt['dir_experiment'], 'samples')
    # Path to where inference-generated images are saved after training, during inference.
    opt['dir_inferences'] = os.path.join(opt['dir_experiment'], 'inferences')

    # Condition to check if existing training data directories should be removed.
    if args.folder_remove:
        if os.path.exists(opt['dir_experiment']):
            shutil.rmtree(opt['dir_experiment'])
        if os.path.exists(opt['dir_tfrecords']):
            shutil.rmtree(opt['dir_tfrecords'])
        if os.path.exists(opt['dir_checkpoints']):
            shutil.rmtree(opt['dir_checkpoints'])
        if os.path.exists(opt['dir_samples']):
            shutil.rmtree(opt['dir_samples'])
        if os.path.exists(opt['dir_inferences']):
            shutil.rmtree(opt['dir_inferences'])

    # Number of batches to process during the training.
    # This can control how long an "epoch" is if dataset is very large.
    total_images = count_images_in_directory(opt['dir_tfrecords'])
    opt['batch_num'] = int(total_images/args.batch_size)

    # Ensure the directories for samples, checkpoints, & inferences exist.
    os.makedirs(opt['dir_experiment'], exist_ok=True)
    os.makedirs(opt['dir_tfrecords'], exist_ok=True)
    os.makedirs(opt['dir_samples'], exist_ok=True)
    os.makedirs(opt['dir_checkpoints'], exist_ok=True)
    os.makedirs(opt['dir_inferences'], exist_ok=True)

    # Return the populated configuration dictionary.
    return opt


In [ ]:
#@title Class - MyGenerativeAdversarialModel
'''
Required libraries:
import os
import glob
import copy
import math
import random
import time
import shutil

import tensorflow as tf
import numpy as np

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *
'''

class MyGenerativeAdversarialModel:
    """
    A class for constructing & training a generative adversarial network (GAN) model for image-to-image translation.
    This class encapsulates both the generator & discriminator components, handling training,
    inference, & utility functions like saving & loading models.

    Attributes:
        opt (dict): Configuration options for various aspects of the GAN.
    """

    def __init__(self, opt: dict):
        """
        Initializes the GAN model with the specified options.

        This constructor method sets up a GAN by initializing its generator & discriminator components using configuration settings
        provided in a dictionary. It stores these settings & constructs the models according to specified parameters,
        making them ready for training or inference.

        Args:
            opt (dict): Configuration dictionary specifying generator & discriminator settings.
                        This dictionary must contain all necessary details such as the size of the latent space,
                        filter sizes, & any other model-specific parameters required for constructing the generator and
                        discriminator models.
        """
        # Store the configuration options in an instance variable for later use throughout the class.
        # This includes settings for both the generator & discriminator which define aspects like architecture depth,
        # learning rates, batch sizes, etc.
        self.opt = opt

        # Initialize the generator model:
        # The generator is created using a method that interprets the settings in `opt` to build the model's layers & configuration.
        # This part of the GAN generates data (e.g., images) from noise.
        self.generator = self.create_generator()


        # Initialize the discriminator model:
        # Similarly, the discriminator is constructed using another method that also uses settings from `opt`.
        # The discriminator's role is to evaluate whether given data is real or generated by the generator.
        self.discriminator = self.create_discriminator()

        # If checkpoints are available, load the weights on the generator & discriminator
        self.model_load()

    def create_generator(self, none_size_condition: bool = False) -> tf.keras.Model:
        """
        Creates the generator model for the GAN.

        Args:
            none_size_condition (bool): If True, allows the input image to have any size. Defaults to False.

        Returns:
            tf.keras.Model: The generator model.
        """
        # Define the input layer with variable size if none_size_condition is True
        if none_size_condition:
            inputs = tf.keras.layers.Input((None, None, 3))  # various sizes (min is 32 for both width & height)
        else:
            # Define the input layer with a fixed size based on the image_size
            inputs = tf.keras.layers.Input((self.opt['image_size'], self.opt['image_size'], 3))

        # Apply image conversion layer
        x = ImageConversion(-1)(inputs)

        # Apply the first generator residual block
        x = generator_residual_block(x, self.opt['generator']['reduce_factor'])

        # # Apply the first scaled convolution layer to get initial RGB output
        x_rgb_1 = ScaledConv2d(channel_count=3, kernel_size=1, padding='same')(x)

        # Apply the final image conversion layer to get the output
        outputs = ImageConversion(0)(x_rgb_1)

        # Return the constructed generator model
        return tf.keras.Model(inputs=inputs, outputs=outputs)

    def create_discriminator(self) -> tf.keras.Model:
        """
        Creates the discriminator model for the GAN.

        Returns:
            tf.keras.Model: The discriminator model.
        """
        # Get the filter counts for resolutions up to the image_size
        filter_counts = {key: item for key, item in self.opt['discriminator']['feature_counts'].items()
                         if key <= self.opt['image_size']}

        # Define input layers for normal & abnormal images
        inputs_normal = tf.keras.layers.Input((self.opt['image_size'], self.opt['image_size'], 3))
        inputs_abnormal = tf.keras.layers.Input((self.opt['image_size'], self.opt['image_size'], 3))

        # Apply image conversion layer to the inputs
        x_normal = ImageConversion(-1)(inputs_normal)
        x_abnormal = ImageConversion(-1)(inputs_abnormal)

        x = tf.keras.layers.concatenate([x_normal, x_abnormal], axis=-1)

        # Initial convolution
        x = ScaledConv2d(channel_count=filter_counts[self.opt['image_size']][0],
                         kernel_size=3, strides=1, padding='same')(x)
        x = ScaledLeakyReLU()(x)

        # Apply discriminator convolution blocks
        for resolution, filter_count in list(filter_counts.items())[:-1]:
            x = discriminator_conv_block(filter_count, x)

        # Apply Minibatch Standard Deviation layer
        x = MinibatchStandardDeviation()(x)

        # Apply final convolution layers
        final_filter_count = list(filter_counts.items())[-1][-1][-1]
        x = ScaledConv2d(channel_count=final_filter_count,
                         kernel_size=3, strides=1, padding='same')(x)
        x = ScaledLeakyReLU()(x)

        x = ScaledConv2d(channel_count=final_filter_count,
                         kernel_size=int(x.shape[1]), strides=1, padding='valid')(x)
        x = ScaledLeakyReLU()(x)

        # Flatten the output & apply a dense layer to get the final output
        x = tf.keras.layers.Flatten()(x)
        outputs = ScaledDense(output_count=1)(x)

        # Return the constructed discriminator model
        return tf.keras.Model(inputs=[inputs_normal, inputs_abnormal], outputs=outputs)

    def reduce_across_batch(self, x: tf.Tensor) -> tf.Tensor:
        """
        Reduces the input tensor across the batch by computing the mean.

        Args:
            x (tf.Tensor): Input tensor to be reduced.

        Returns:
            tf.Tensor: Scalar tensor obtained by reducing `x`.
        """
        return tf.reduce_sum(x) / self.opt['batch_size']

    def parse_function(self, example_proto: tf.Tensor) -> tf.Tensor:
        """
        Parse & process data from a TFRecord file.

        Args:
            example_proto (tf.Tensor): A tensor containing a serialized TFRecord example.

        Returns:
            tf.Tensor: A tensor containing a processed image.
        """
        # Define the features in the TFRecord that are to be extracted.
        feature_description = {
            'image_raw': tf.io.FixedLenFeature([], tf.string),
        }
        # Parse the input `tf.train.Example` proto using the dictionary above.
        example = tf.io.parse_single_example(example_proto, feature_description)
        # Decode the image, assume RGB channels.
        image = tf.io.decode_png(example['image_raw'], channels=3)
        # Convert the image to floating point values & normalize the image to the range [0, 255].
        image = tf.cast(image, tf.float32)

        # Generate random noise
        noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=self.opt['noise_std'])

        # Scale the noise by the noise_factor
        noise_factor = np.random.rand(1)/4 + 0.2
        noise = noise * noise_factor

        # Add noise to the image
        noisy_image = image + noise

        # Clip the values to be in the valid range [0, 255] for RGB images
        noisy_image = tf.clip_by_value(noisy_image, 0.0, 255.0)

        # Normalize images to [0, 1] for model compatibility
        noisy_image_normalized = noisy_image / 255.0
        image_normalized = image / 255.0

        # Return the normal & abnormal images.
        return noisy_image_normalized, image_normalized  # abnormal, normal

    def load_dataset(self) -> tf.data.Dataset:
        """
        Load & parse the dataset from specified TFRecord files.

        Returns:
            tf.data.Dataset: A TensorFlow Dataset object containing processed images.
        """
        # Get all available tfrecords
        tfrecord_paths = tf.io.gfile.glob(os.path.join(self.opt['dir_tfrecords'], '*.tfrecord'))

        # Create a dataset from the file paths.
        dataset = tf.data.TFRecordDataset(tfrecord_paths)

        # Map parsing function across dataset elements with parallel processing.
        dataset = dataset.map(self.parse_function, num_parallel_calls=tf.data.experimental.AUTOTUNE)

        # # Shuffle the dataset using the buffer size specified in the configuration.
        # dataset = dataset.shuffle(self.opt['buffer_size'])

        # Batch the dataset with the specified batch size from the configuration.
        dataset = dataset.batch(self.opt['batch_size'])

        # Take the specified number of batches to ensure each epoch has a consistent number of batches.
        dataset = dataset.take(self.opt['batch_num'])

        # Prefetch the dataset to improve training efficiency.
        dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

        # Return the fully prepared dataset.
        return dataset

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def generator_step(self, abnormal_images: tf.Tensor) -> tf.Tensor:
        """
        Performs a training step for the generator.

        Args:
            abnormal_images (tf.Tensor): Batch of normal images.

        Returns:
            tf.Tensor: The loss of the generator.
        """
        # Generate fake normal images using the generator
        fake_normal_images = self.generator(abnormal_images, training=True)

        # Classify the fake normal images using the discriminator
        fake_classifications = self.discriminator([abnormal_images, fake_normal_images], training=False)

        # Calculate the loss for the generator
        loss = self.reduce_across_batch(tf.nn.softplus(-fake_classifications))

        # Compute the gradients of the loss with respect to the generator's trainable variables.
        grads = tf.gradients(loss, self.generator.trainable_variables)

        # Apply the calculated gradients to the generator's trainable variables using its optimizer.
        # This step updates the weights of the generator to improve its performance.
        self.generator.optimizer.apply_gradients(zip(grads, self.generator.trainable_variables))

        # Return the loss computed in this step, providing feedback on how well the generator is performing.
        return loss

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def discriminator_step(self, abnormal_images: tf.Tensor, normal_images: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Performs a training step for the discriminator.

        Args:
            abnormal_images (tf.Tensor): Batch of abnormal images.
            normal_images (tf.Tensor): Batch of normal images.

        Returns:
            Tuple[tf.Tensor, tf.Tensor, tf.Tensor]: The total loss, real loss, & fake loss.
        """
        # Generate fake normal images using the generator
        fake_normal_images = self.generator(abnormal_images, training=False)

        # Classify real normal images
        real_classifications = self.discriminator([abnormal_images, normal_images], training=True)

        # Classify fake normal images
        fake_classifications = self.discriminator([abnormal_images, fake_normal_images], training=True)

        # Calculate the loss for real images using softplus, designed to penalize the discriminator
        # when it fails to classify real images as real.
        real_loss = self.reduce_across_batch(tf.nn.softplus(-real_classifications))

        # Calculate the loss for fake images using softplus, designed to penalize the discriminator
        # when it fails to classify fake images as fake.
        fake_loss = self.reduce_across_batch(tf.nn.softplus(fake_classifications))

        # Calculate the total discriminator loss as the sum of real & fake losses.
        d_loss = real_loss + fake_loss

        # Compute gradients of the total loss with respect to the discriminator’s trainable variables.
        d_grads = tf.gradients(d_loss, self.discriminator.trainable_variables)

        # Apply the computed gradients to the discriminator's variables using its optimizer.
        # This updates the discriminator to improve its classification accuracy.
        self.discriminator.optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))

        # Return the total loss, real loss, & fake loss, providing insight into how well the discriminator is learning
        # to distinguish between real & fake images.
        return d_loss, real_loss, fake_loss

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def regularization_step(self, abnormal_images: tf.Tensor, normal_images: tf.Tensor) -> tf.Tensor:

        # Classify real images using the discriminator; `training=True` enables updates to batch normalization layers based on these images.
        real_classifications = self.discriminator([abnormal_images, normal_images], training=True)

        # Compute gradients of the discriminator's classifications with respect to the input images.
        # This gradient reflects how changes in input images affect the discriminator's decision-making process.
        real_grads = tf.gradients(tf.reduce_sum(real_classifications), normal_images)

        # Compute the L2 norm of the gradients for each image, then average across the batch.
        # Squaring & summing the components of gradients for each image helps measure how "steep" these gradients are.
        gradient_loss = self.reduce_across_batch(tf.reduce_sum(tf.square(real_grads), axis=[1, 2, 3]))

        # Calculate the gradient penalty strength, scaling it by a factor from the configuration
        # & typically involves a hyperparameter to balance the regularization strength.
        gradient_penalty_strength = 10. * 0.5 * self.opt['discriminator']['regularization_interval']

        # Multiply the average gradient norm by the penalty strength to compute the total gradient penalty.
        gradient_penalty = gradient_loss * gradient_penalty_strength

        # Compute gradients of the gradient penalty with respect to the discriminator's trainable variables.
        reg_grads = tf.gradients(gradient_penalty, self.discriminator.trainable_variables)

        # Set the last gradients to zero. This can be a specific technique to avoid regularizing certain components
        # or due to specific architectural needs.
        reg_grads[-1] = tf.zeros_like(self.discriminator.trainable_variables[-1])

        # Apply these computed gradients to the discriminator's variables to update them in a way that minimizes the gradient penalty.
        self.discriminator.optimizer.apply_gradients(zip(reg_grads, self.discriminator.trainable_variables))

        # Return the computed gradient penalty as a scalar tensor, which quantifies the enforced smoothness of the discriminator's function.
        return gradient_penalty

    def plot(self, abnormal_images: tf.Tensor, normal_images: tf.Tensor) -> None:
        """
        Plots & saves a comparison of abnormal, generated normal, & true normal images.

        Args:
            abnormal_images (tf.Tensor): Tensor containing abnormal images.
            normal_images (tf.Tensor): Tensor containing normal images.
        """
        # Generate normal images using the generator model
        fake_normal_images = self.generator(abnormal_images, training=False)

        # Get target dimensions for resizing abnormal images
        target_height, target_width = int(normal_images.shape[1]), int(normal_images.shape[2])

        # Randomly select 5 images from the batch for plotting
        num = np.random.permutation(self.opt['batch_size'])[:5]

        # Concatenate the abnormal, generated normal, & true normal images
        image = tf.concat(
            [
                tf.concat(
                    [
                        tf.image.resize(abnormal_images[i, ...], (target_height, target_width), method=tf.image.ResizeMethod.NEAREST_NEIGHBOR),
                        fake_normal_images[i, ...],
                        normal_images[i, ...]
                    ],
                    axis=1
                )
                for i in num
            ],
            axis=0
        )

        # Define the filename & path where the grid image will be saved
        filename = os.path.join(self.opt['dir_samples'], f'samples_{int(time.time())}.png')

        # Remove old samples if there are at least 10 samples in the directory
        previous_images = glob.glob(os.path.join(self.opt['dir_samples'], '*.png'))
        if len(previous_images) >= 10:
            for file_path in previous_images:
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")

        # Save the final image grid to the specified filename
        tf.keras.utils.save_img(filename, image)

    def model_save(self):
        """
        Saves the weights of the generator & discriminator models.

        This method provides functionality to persist the current state of the generator & discriminator models by saving their
        weights. This is crucial for checkpointing the models during training, allowing for recovery & continuation of training
        at a later time, or using the trained models for inference without needing to retrain.

        """
        # Define the filename & path for saving the generator's weights.
        # The directory for storing checkpoints is specified in the configuration options under 'dir_checkpoints'.
        filename = os.path.join(self.opt['dir_checkpoints'], 'generator.h5')
        # Save the weights of the generator model to the specified file.
        # The .h5 file format is used here, which is a common storage format for model weights in Keras.
        self.generator.save_weights(filename)

        # Define the filename & path for saving the discriminator's weights.
        filename = os.path.join(self.opt['dir_checkpoints'], 'discriminator.h5')
        # Save the weights of the discriminator model to the specified file.
        # Saving the discriminator's weights is equally important as it allows for the entire GAN architecture to be restored.
        self.discriminator.save_weights(filename)

    def training(self):
        """
        Conducts the training process for the GAN, including both generator & discriminator components.

        This method manages the entire training lifecycle of a GAN. It sets up optimizers for both models,
        loads any existing model weights, & iteratively trains the generator & discriminator through a series
        of epochs. It includes checkpoints & plotting intervals for monitoring & saving progress, & applies
        regularization to ensure the stability of the training process.

        """
        # Set up the Adam optimizer for the generator with specified learning rate & beta coefficients.
        # These parameters are pulled from the configuration dictionary (`self.opt`).
        self.generator.optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.opt['generator']['lr'],
            beta_1=self.opt['generator']['beta1'],
            beta_2=self.opt['generator']['beta2']
        )

        # Set up the Adam optimizer for the discriminator.
        # The learning rate & beta coefficients are scaled by a 'lazy_ratio' from the configuration,
        # which is used to adjust the training speed relative to the generator.
        self.discriminator.optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['lr'],
            beta_1=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['beta1'],
            beta_2=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['beta2']
        )

        # Load previously saved model weights if available, enabling continuation of training from a checkpoint.
        self.model_load()

        # Create a dataset. This dataset includes preprocessing & batching of images.
        dataset = self.load_dataset()

        # Begin training over the specified number of epochs.
        for epoch in range(self.opt['epochs']):

            # Initiate batch numbering
            batch   = 0
            # Initiate running loss for both generator & discriminator
            running_d_loss = 0
            running_g_loss = 0

            # Initiate a progress bar using tqdm library
            pbar = tqdm(dataset, total=self.opt['batch_num'], desc=f"Epoch: {epoch+1:04d}", ncols=150,leave = False)

            for images_abnormal, images_normal in pbar:

                # Control if we do not encounter & incomplete final batch
                if len(images_normal) == self.opt['batch_size']:
                    # Perform a training step for the discriminator & capture the loss.
                    d_loss, _, _ = self.discriminator_step(images_abnormal, images_normal)

                    # Regularly apply gradient penalty as a regularization method to stabilize the discriminator,
                    # based on the specified interval in the configuration.
                    if (batch + 1) % self.opt['discriminator']['regularization_interval'] == 0:
                        _ = self.regularization_step(images_abnormal, images_normal)

                    # Perform a training step for the generator & capture the resulting loss.
                    g_loss = self.generator_step(images_abnormal)

                    # At specified intervals, generate & save a grid of images to visually monitor the generator's progress.
                    if (batch + 1) % self.opt['plot_intervals'] == 0:
                        self.plot(images_abnormal, images_normal)

                    # Save the model at specified intervals, creating checkpoints for long-running training processes.
                    if (batch + 1) % self.opt['checkpoint_intervals'] == 0:
                        self.model_save()

                    # Capture the running losses
                    running_d_loss += d_loss
                    running_g_loss += g_loss


                    # Add running loss information to the progress bar.
                    pbar.set_postfix(loss=f"Discriminator Loss: {running_d_loss/(batch+1):10.8f} | Generator Loss: {running_g_loss/(batch+1):10.8f}")

                # update the batch
                batch += 1

    def model_load(self):
        """
        Loads the weights of the generator & discriminator models if available.

        This method checks for the presence of saved weight files for both the generator & discriminator models & loads them
        if they exist. Loading model weights is essential for resuming training from a checkpoint, deploying models for inference,
        or continuing experiments without needing to retrain models from scratch.

        """



        try:
            # Define the filename & path for the generator's saved weights.
            # The directory where checkpoints are stored is specified in the configuration options under 'dir_checkpoints'.
            filename = os.path.join(self.opt['dir_checkpoints'], 'generator.h5')
            # Check if the file with the saved weights exists.
            if os.path.exists(filename):
                # Load the weights into the generator model if the file exists.
                # This restores the generator's state to the last checkpointed version, enabling continuity in training or deployment.
                self.generator.load_weights(filename)

            # Define the filename & path for the discriminator's saved weights.
            filename = os.path.join(self.opt['dir_checkpoints'], 'discriminator.h5')
            # Check if the file with the saved weights exists.
            if os.path.exists(filename):
                # Load the weights into the discriminator model if the file exists.
                # Similarly to the generator, this restores the discriminator's state to the last checkpointed version.
                self.discriminator.load_weights(filename)
        except:
            pass

    def translation(self, image_file: str) -> None:
        """
        Applies an image-to-image translation to an input image using the generator model & saves the result.

        Args:
            image_file (str): Path to the input image file.
        """
        # Extract the base name & directory of the image file
        image_basename = os.path.basename(image_file).split('.')[0]
        image_dir = os.path.dirname(image_file)

        # Read the abnormal image using OpenCV & convert it from BGR to RGB format
        image = cv2.imread(image_file)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize the image & expand dimensions to match the model's input shape
        image_abnormal = np.expand_dims(image / 255.0, axis=0)

        # Generate the normal image using the generator model
        image_normal = self.generator(image_abnormal)

        # Construct the filename for the normal image
        filename = os.path.join(image_dir, f"{image_basename}_normal.png")

        # Save the normal image using TensorFlow's utility function
        tf.keras.utils.save_img(filename, image_normal[0, ...])

    def image_to_image_translation(self, path: Union[str, bytes]) -> None:
        """
        Translate images for a given file or directory of files.

        Args:
            path (Union[str, bytes]): Path to the input image file or directory containing image files.
        """
        # Create the generator model with variable input size
        self.generator = self.create_generator(none_size_condition=True)

        # Load the pre-trained weights for the generator
        filename = os.path.join(self.opt['dir_checkpoints'], 'generator.h5')
        self.generator.load_weights(filename)

        print('Generator model loaded.')

        # Check if the provided path is a file
        if os.path.isfile(path):
            # Apply image-to-image translation to the single image file
            self.translation(path)
        elif os.path.isdir(path):
            # Get all image files in the directory
            image_files = sorted(glob.glob(os.path.join(path, '*.png')))
            for image_file in image_files:
                # Apply image-to-image translation to each image file
                self.translation(image_file)
                print(f"Completed: {image_file}")
        else:
            print("\nThe provided path is not valid!\n")


In [ ]:
#@title Main Run - Training
# Let's connect to Google Drive to manage our files efficiently.
# The free version of Google Drive gives us 15GB of disk space as of May 2024.
# Follow the instructions to give access to Google Drive & then comeback to this tab.
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os

# Get the input arguments
args = Args()

import glob
import copy
import math
import random
import time
import shutil
import requests
import cv2


import tensorflow as tf
import numpy as np
import pandas as pd

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *

# Suppress TensorFlow warnings
# Here's how you can set TF_CPP_MIN_LOG_LEVEL to different levels:
# 0: Default, shows all logs (DEBUG, INFO, WARN, ERROR, & FATAL).
# 1: Filters out INFO logs.
# 2: Additionally filters out WARNING logs.
# 3: Additionally filters out ERROR logs, showing only FATAL.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Execute the operational logic of the script initializing data, models, and
# performing training.

# Initialize with options
opt    = initiate(args)

# Download the data.
# You can use the records of any image data repository with minor changes to the code.
# Here, we have the records of animal faces.
# Source: https://www.kaggle.com/datasets/dimensi0n/afhq-512.
# Let's download the data using the' "wget" Linux command.
# First, make sure that the data is not already in the Google Drive designated folder.
tfrecords_path = glob.glob(os.path.join(opt['dir_tfrecords'], '*'))
if len(tfrecords_path) == 0:
    !wget https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_{args.image_size}.zip
    # Unzip the files into the tfrecords folder.
    !unzip animal_faces_{args.image_size}.zip -d {opt['dir_tfrecords']}
    # remove the zip file
    !rm animal_faces_{args.image_size}.zip
    # Delay for Google Drive usual time lags.
    # If the cell runs without any training, you need to run this cell in a few minutes (15 minutes)
    # The reason is sometimes it takes time for Colab to be fully synched with Google Drive when
    # we copy files over there. For larger files (image_size), you need to increase this delay.
    time.sleep(900)

    # reboot the instance
    !sudo reboot

# Create the GAN object.
my_obj = MyGenerativeAdversarialModel(opt)

# Print generator model summary.
# my_obj.generator.summary()
tf.keras.utils.plot_model(
    my_obj.generator,
    to_file='model_generator.png',
    show_shapes=True,
    show_layer_names=True,
    dpi=600)

# Print discriminator model summary.
# my_obj.discriminator.summary()
tf.keras.utils.plot_model(
    my_obj.discriminator,
    to_file='model_discriminator.png',
    show_shapes=True,
    show_layer_names=True,
    dpi=600)

# We are not going to use CPU to train these massive models
if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    with tf.device(gpus[0].name.replace('/physical_','')):
        # Train with one (first) GPU.
        my_obj.training()


In [ ]:
#@title Inference
#@title Main Run - Training
# Let's connect to Google Drive to manage our files efficiently.
# The free version of Google Drive gives us 15GB of disk space as of May 2024.
# Follow the instructions to give access to Google Drive & then comeback to this tab.
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os

# Get the input arguments
args = Args()

import glob
import copy
import math
import random
import time
import shutil
import requests


import tensorflow as tf
import numpy as np
import pandas as pd

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *

# Suppress TensorFlow warnings
# Here's how you can set TF_CPP_MIN_LOG_LEVEL to different levels:
# 0: Default, shows all logs (DEBUG, INFO, WARN, ERROR, & FATAL).
# 1: Filters out INFO logs.
# 2: Additionally filters out WARNING logs.
# 3: Additionally filters out ERROR logs, showing only FATAL.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Execute the operational logic of the script initializing data, models, and
# performing training or inference based on the mode.

# Initialize with options
opt    = initiate(args)

inference_path = glob.glob(os.path.join(opt['dir_inferences'], '*'))
if len(inference_path) == 0:
    !wget https://storage.googleapis.com/535743/gan/animal_faces/inference_examples_denoising.zip
    !unzip inference_examples_denoising.zip -d {opt['dir_inferences']}
    !rm inference_examples_denoising.zip
    time.sleep(5)


my_obj = MyGenerativeAdversarialModel(opt)


if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    with tf.device(gpus[0].name.replace('/physical_','')):
        # Inference with one (first) GPU.
        my_obj.image_to_image_translation(opt['dir_inferences'])
else:
    cpus = tf.config.list_physical_devices('CPU')
    with tf.device(cpus[0].name.replace('/physical_','')):
        # Inference with one (first) GPU
        my_obj.image_to_image_translation(opt['dir_inferences'])

# <font color="#418FDE" size="6.5" uppercase>**C: Generative Adversarial Networks (GANs) for Image-to-Image Translation**</font>
----

In this lecture, you learned to:

* Develop a Generative Adversarial Network (GAN) for an image-to-image translation experiment.


In the next Module (Module 7), we will go over "Introduction to PyTorch".